# FFT in Practice

This notebook extracts the practical FFT material from the legacy notebooks: windowing, frequency resolution, and spectrograms. The goal is to show why FFT settings change what you think you are seeing.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## Resolution and Leakage

If a tone does not fit perfectly into the observation window, its energy leaks into neighboring FFT bins. Window functions trade peak sharpness for lower sidelobes.

In [ ]:
fs = 48_000
duration = 0.03
t = np.arange(0, duration, 1 / fs)
tone = np.cos(2 * np.pi * 1037 * t)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
plot_waveform(tone[:1200], fs=fs, ax=axes[0], title="Off-Bin Tone")

for name in ["boxcar", "hann", "hamming", "blackman"]:
    freqs, mags_db = power_spectrum(tone, fs=fs, nfft=8192, window=name)
    axes[1].plot(freqs, mags_db, label=name)

axes[1].set_xlim(700, 1400)
axes[1].set_ylim(-140, 5)
axes[1].set_title("Window Comparison")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Magnitude (dB)")
axes[1].legend()
plt.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))

def update_window(window="hann", nfft=4096):
    ax.clear()
    freqs, mags_db = power_spectrum(tone, fs=fs, nfft=nfft, window=window)
    ax.plot(freqs, mags_db, color="tab:orange")
    ax.set_xlim(700, 1400)
    ax.set_ylim(-140, 5)
    ax.set_title(f"{window} window, nfft={nfft}")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Magnitude (dB)")
    fig.canvas.draw_idle()

controls = widgets.interactive(
    update_window,
    window=dropdown(options=["boxcar", "hann", "hamming", "blackman"], value="hann", description="Window"),
    nfft=int_slider(min_value=1024, max_value=16384, step=1024, value=4096, description="NFFT"),
)
display(controls)


## Spectrograms

The spectrogram section in `dsp.ipynb` is the short-time Fourier transform in action. Each vertical slice is an FFT over a short window; stacked together, those FFTs show how frequency content evolves over time.

In [ ]:
fs = 48_000
t = np.arange(0, 1.5, 1 / fs)
chirp = signal.chirp(t, f0=300, f1=6000, t1=t[-1], method="linear")
am_chirp = am_modulate(chirp, carrier_freq=10_000, fs=fs, mod_index=0.7)
fm_chirp = fm_modulate(chirp, carrier_freq=10_000, fs=fs, freq_dev=2500)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_spectrogram(am_chirp, fs=fs, ax=axes[0], title="AM Chirp Spectrogram")
plot_spectrogram(fm_chirp, fs=fs, ax=axes[1], title="FM Chirp Spectrogram")
for ax in axes:
    ax.set_ylim(0, 15_000)
plt.tight_layout()


## What to Try

- Compare `boxcar` and `blackman` windows and look at how the sidelobe floor changes.
- Double `nfft` and notice that the line positions become smoother, but the underlying data did not gain new information.
- Change the chirp start and end frequencies and watch the spectrogram slope change.

## Key Takeaway

The FFT is not just "run it and trust it." Window choice, FFT length, and segment size all change the picture you get, and radio tools depend on choosing those parameters deliberately.